# TraceX LLM runs on Kaggle (Member 3)

Runs Qwen2.5-7B-Instruct and Qwen2.5-3B-Instruct (4-bit) for experiments E1-E3.
**Both models run on the same single GPU** so accuracy and speed are comparable.

**Before you start (once):** verify your phone number at https://www.kaggle.com/settings.

**Notebook settings (right panel -> Session options), every time:**
1. Accelerator: **GPU T4 x2**. The notebook pins everything to **one** T4 (GPU 0) so every run uses identical hardware.
2. Internet: **On**.

**Workflow (one run does everything): `STAGE = "all"`**
1. Leave `STAGE = "all"` in the next cell and click **Save Version -> Save & Run All (Commit)**. It runs in the
   background (Kaggle limit 12 h) and keeps the output. You can close the browser.
2. `all` runs both models in turn on the same T4: for each model a 20-instance debug run, a sanity check
   (a model that fails it is skipped, never half-run), then E1 (NYC + TKY) and E3 (Massive-STEPS names test).
   Every run already in the repo or in an attached results zip is skipped. Section 5 prints what is left first.
3. One failed run does not stop the others; the failures are listed at the end.
4. At the end it runs the evaluation and figures (`tracex_llm.evaluate`, `tracex_llm.report`) and prints the tables.
5. Download `tracex_llm_results.zip` from the **Output** tab (results + reports) and give it to Member 3.

Single stages still work: set `STAGE` to `debug`, `gate1`, `e1` or `e3` and `MODEL` to one model.

**Resuming after a timeout:** upload the last `tracex_llm_results.zip` as a Kaggle Dataset (any name), add it to
the notebook with *Add Input*, and run again. Finished instances are skipped.

In [ ]:
# ---- settings ----------------------------------------------------------------
STAGE = "all"      # "all" (everything, both models) | "debug" | "gate1" | "e1" | "e3"
MODEL = "qwen7b"   # used only for a single stage: "qwen7b" | "qwen3b"
ALL_MODELS = ["qwen7b", "qwen3b"]   # order used by STAGE = "all"
# -----------------------------------------------------------------------------
REPO = "https://github.com/CipherEnigma/TraceX.git"
BRANCH = "Aayush"          # change to "main" once the LLM code is merged
REPO_DIR = "/tmp/TraceX"   # outside /kaggle/working so the output holds only the results zip
OUT_ZIP = "/kaggle/working/tracex_llm_results.zip"

## 1. GPU: pin to one T4

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # must be set before torch is imported
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 2. Code and dependencies

If the repo is **private**, add a read-only GitHub token as a Kaggle secret named `GITHUB_TOKEN`
(*Add-ons -> Secrets*). Never paste the token into a cell. A public repo needs no token.

In [ ]:
import subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"   # fail fast instead of asking for a username

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass   # no secret attached: fine for a public repo
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).replace(token, "***") if token else (r.stderr or r.stdout)
        hint = ("" if token else "\nIf the repo is private, add a GITHUB_TOKEN Kaggle secret (see above). "
                "Also check Internet is On in Session options.")
        raise RuntimeError(f"git {args[0]} failed:\n{msg}{hint}")   # the token never appears here
    return r.stdout.strip()

if os.path.isdir(REPO_DIR):
    git("-C", REPO_DIR, "fetch", url, f"{BRANCH}:refs/remotes/origin/{BRANCH}")
    git("-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}")
else:
    git("clone", "--branch", BRANCH, url, REPO_DIR)
    git("-C", REPO_DIR, "remote", "set-url", "origin", REPO)   # do not keep the token in .git/config
COMMIT = git("-C", REPO_DIR, "rev-parse", "--short", "HEAD")
print("code at commit", COMMIT, "(token used)" if token else "(public clone)")

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-llm.txt
import importlib.metadata as md
for p in ["torch", "transformers", "accelerate", "bitsandbytes", "pandas"]:
    print(p, md.version(p))

## 3. Data (Member 1's pipeline, about 2 minutes)
The pipeline is deterministic. The check below fails if the evaluation samples differ from the ones committed in the repo.

In [ ]:
os.chdir(REPO_DIR)
!python -m tracex_data all
diff = subprocess.run(["git", "status", "--porcelain", "--", "data/processed"], capture_output=True, text=True).stdout
assert not diff.strip(), f"processed samples differ from the repo:\n{diff}"
print("samples identical to the committed ones")

## 4. Restore earlier results (resume)

In [ ]:
import glob, shutil, zipfile, pathlib, sys
sys.path.insert(0, REPO_DIR)
from tracex_llm import config as C

restored = 0
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    with zipfile.ZipFile(z) as f:
        for n in f.namelist():
            dest = pathlib.Path(REPO_DIR) / n
            if n.startswith("results/llm/") and not n.endswith("/") and not dest.exists():
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(f.read(n)); restored += 1
for src in glob.glob("/kaggle/input/**/results/llm/**/*.*", recursive=True):  # unzipped uploads
    rel = src.split("results/llm/", 1)[1]
    dest = C.RESULTS_DIR / rel
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dest); restored += 1
print(f"restored {restored} file(s)")

## 5. Plan: what is already done and what this run will do
| Stage | What | Instances per model |
|---|---|---|
| `debug` | NYC, both methods, 20 instances: speed + sanity check | 2 x 20 |
| `gate1` | NYC, both methods, full 500 sample: reproduction check (Gate 1) | 2 x 500 |
| `e1` | NYC + TKY, both methods (E1 and E2 come from the same records) | 4 x 500 |
| `e3` | Massive-STEPS Tokyo, both methods, names present / removed / aliased | 6 x 334 |
| `all` | `debug` + `e1` + `e3` for every model in `ALL_MODELS` | 4,044 |

In [ ]:
import json
from tracex_data.api import load_dataset
from tracex_llm.run import Session, run_one, summarize, load_done

STAGES = {
    "debug": [("nyc", m, "none", "sample_test_debug20") for m in C.METHODS],
    "gate1": [("nyc", m, "none", "sample_test_500") for m in C.METHODS],
    "e1":    [(d, m, "none", "sample_test_500") for d in ("nyc", "tky") for m in C.METHODS],
    "e3":    [("msteps_tokyo", m, c, "sample_test_500") for m in C.METHODS for c in C.NAME_CONDITIONS],
}
STAGES["all"] = STAGES["debug"] + STAGES["e1"] + STAGES["e3"]
MODELS_TO_RUN = ALL_MODELS if STAGE == "all" else [MODEL]
SEC_PER_INST = {"qwen7b": 3.9, "qwen3b": 2.5}   # 7B measured on a T4; 3B is a guess until its debug run

_n = {}
def expected(dataset, sample):
    if (dataset, sample) not in _n:
        _n[(dataset, sample)] = len(load_dataset(dataset).instances(sample))
    return _n[(dataset, sample)]

left_h = 0.0
print(f"{'run':80s} {'done':>9s}")
for model in MODELS_TO_RUN:
    for d, m, c, s in STAGES[STAGE]:
        p = C.result_path(d, model, m, c, None, s)
        done, exp = len(load_done(p)), expected(d, s)
        left_h += max(exp - done, 0) * SEC_PER_INST[model] / 3600
        print(f"{p.stem:80s} {done:4d}/{exp:<4d} {'ok' if done >= exp else ''}")
print(f"\nabout {left_h:.1f} GPU-hours of generation left (Kaggle limit per run: 12 h)")

## 6. Run
Each model is loaded once. Results are saved to the zip after every run, so a timeout never loses finished work.
In `all` mode a model whose 20-instance debug run gives more than 20% invalid answers is skipped
(something is wrong with it; running it for hours would waste the GPU quota).

In [ ]:
import gc, time, traceback, torch

SANITY_MAX_INVALID = 0.20
OUT_DIRS = {"results/llm": C.RESULTS_DIR, "reports/llm": pathlib.Path(REPO_DIR) / "reports/llm"}

def save_zip():
    with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for prefix, root in OUT_DIRS.items():
            for f in (sorted(root.rglob("*")) if root.exists() else []):
                if f.is_file():
                    z.write(f, prefix + "/" + f.relative_to(root).as_posix())
    print(f"[saved] {OUT_ZIP}")

def fmt(r):
    return ", ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in (r or {}).items())

failures, skipped_models, t_all = [], [], time.time()
for model in MODELS_TO_RUN:
    print(f"\n{'=' * 30} {model} {'=' * 30}")
    sess = Session(model)
    for dataset, method, condition, sample in STAGES[STAGE]:
        try:
            r = run_one(sess, dataset, method, condition, None, sample)
        except Exception:
            traceback.print_exc()
            failures.append((model, dataset, method, condition, sample))
            gc.collect(); torch.cuda.empty_cache()
            continue
        finally:
            save_zip()
        print(f"  -> {model} {dataset} {method} {condition} {sample}: {fmt(r)}")
        if STAGE == "all" and sample == "sample_test_debug20" and r and r["invalid_rate"] > SANITY_MAX_INVALID:
            print(f"!! {model}: {r['invalid_rate']:.0%} invalid answers on the debug run -> skipping {model}")
            skipped_models.append(model)
            break
    del sess
    gc.collect(); torch.cuda.empty_cache()   # free the GPU before the next model

print(f"\nfinished in {(time.time() - t_all) / 3600:.2f} h")
print("failed runs:", failures or "none")
print("skipped models:", skipped_models or "none")

## 7. Evaluation and figures (same code as on the laptop, so the numbers match)

In [ ]:
!python -m tracex_llm.evaluate > /tmp/evaluate.log && echo "evaluation written" || tail -20 /tmp/evaluate.log
!python -m tracex_llm.report
save_zip()

## 8. Speed: measured seconds per instance

In [ ]:
rows = []
for meta in sorted(C.RESULTS_DIR.rglob("*.meta.json")):
    m = json.loads(meta.read_text(encoding="utf-8"))
    for s in m["sessions"]:
        if "sec_per_instance" in s:
            rows.append((m["run"], s["backend"].get("gpu"), s["todo"], s["sec_per_instance"]))
for r in rows:
    print(f"{r[0]:80s} {r[1]}  n={r[2]:4d}  {r[3]:.2f} s/inst")
for model in C.MODELS:
    mine = [(r[2], r[3]) for r in rows if f"__{model}__" in r[0]]
    if mine:
        spi = sum(n * s for n, s in mine) / sum(n for n, _ in mine)   # weighted by instances
        print(f"{model}: mean {spi:.2f} s/instance over {sum(n for n, _ in mine)} timed instances")

## 9. Gate 1: reproduction check (NYC)
Brief: our 7B Accuracy@1 should land near the published 0.18-0.23 (Nishida et al.).
More than about 3 points below means a pipeline bug.

In [ ]:
PUBLISHED = {"nyc": (0.18, 0.23), "tky": (0.16, 0.19)}
freq = json.loads((pathlib.Path(REPO_DIR) / "reports/retrieval/nyc_results.json").read_text())["prediction"]["frequency"]
print(f"frequency heuristic (Member 2), NYC sample_test_500: acc1={freq['acc1']:.3f}\n")
lo, hi = PUBLISHED["nyc"]
for model in MODELS_TO_RUN:
    for sample in ("sample_test_debug20", "sample_test_500"):
        for method in C.METHODS:
            p = C.result_path("nyc", model, method, sample=sample)
            if not p.exists():
                continue
            r = summarize(p.stem, p)
            verdict = ("within" if lo <= r["acc1"] <= hi else "ABOVE" if r["acc1"] > hi else
                       "below, but within 3 points" if r["acc1"] >= lo - 0.03 else "MORE THAN 3 POINTS BELOW")
            note = " (20 instances: too few to judge)" if sample.endswith("debug20") else ""
            print(f"{p.stem:70s} n={r['n']:3d} acc1={r['acc1']:.3f} invalid={r['invalid_rate']:.3f} "
                  f"copy={r['copy_rate']:.3f}\n    published {lo:.2f}-{hi:.2f}: {verdict}{note}")

## 10. Look at a few raw answers

In [ ]:
for model in MODELS_TO_RUN:
    for p in sorted(C.RESULTS_DIR.rglob(f"*__{model}__*debug20.jsonl"))[:2]:
        print("==", p.stem)
        for line in p.read_text(encoding="utf-8").splitlines()[:2]:
            r = json.loads(line)
            print(f"  target={r['target']} ranked={r['parse']['ranked'][:5]} status={r['parse']['status']} "
                  f"copied={r['parse']['copied_top1']}\n  raw: {r['raw'][:200]!r}")

## 11. Final results table

In [ ]:
from IPython.display import Markdown, display
md_path = pathlib.Path(REPO_DIR) / "reports/llm/results.md"
display(Markdown(md_path.read_text(encoding="utf-8")) if md_path.exists() else "no results.md: see section 7")
print("Download tracex_llm_results.zip from the Output tab.")